# Dialog Finance: Customer Data Cleaning
**Step 1 of the assessment.** This notebook cleans the raw dataset and produces two files:
- `cleaned_customers.csv`: analysis-ready data (input to tiering, dashboard and deck)
- `data_quality_log.csv`: an audit trail of every issue found and what was done about it

**Principle:** never silently change data. Every fix is logged, and each affected customer row carries a `DQ_Flags` note.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

INPUT = "/mnt/user-data/uploads/20260527_DialogFinance_Assessment_Dataset_F.xlsx"  # change to your local path
OUT = Path("/mnt/user-data/outputs")                                                # change to your local folder

raw = pd.read_excel(INPUT, header=1)   # row 0 is a group-header band, real headers are on row 1
df = raw.copy()
print("Raw shape:", raw.shape)

LOG = []
def record(step, issue, column, rows, action, ids=None):
    LOG.append({"step": step, "issue": issue, "column": column, "rows_affected": int(rows),
                "action_taken": action, "example_customer_ids": ", ".join(map(str, list(ids)[:5])) if ids is not None else ""})

## 1. Duplicates

In [ ]:
# 1a. Exact duplicate rows
m = df.duplicated(keep="first")
record("1a","Exact duplicate record","(all columns)",m.sum(),"Dropped the repeated copy",df.loc[m,"Customer ID"])
df = df[~m]

# 1b. Same Customer ID but a different person (ID collision)
m = df["Customer ID"].duplicated(keep="first")
record("1b","Customer ID reused for a different person","Customer ID",m.sum(),
       "Kept first occurrence, dropped later conflicting record (cannot tell which is correct)",df.loc[m,"Customer ID"])
df = df[~m].reset_index(drop=True)

df["DQ_Flags"] = ""
def flag(mask, text): df.loc[mask,"DQ_Flags"] += text + "; "
print("Rows after de-duplication:", len(df))

## 2. Standardise inconsistent coding

In [ ]:
# Gender
g = df["Gender"].astype(str).str.strip().str.lower().map({"m":"Male","male":"Male","f":"Female","female":"Female"})
m = g != df["Gender"]
record("2a","Inconsistent gender coding (F, m, MALE...)","Gender",m.sum(),"Standardised to Male / Female",df.loc[m,"Customer ID"])
flag(m,"Gender standardised"); df["Gender"] = g

# Credit card flag
c = df["Credit Card"].astype(str).str.strip().str.lower().map(
    {"yes":"Yes","y":"Yes","true":"Yes","1":"Yes","no":"No","n":"No","false":"No","0":"No"})
m = c != df["Credit Card"]
record("2b","Inconsistent Yes/No coding (Y, TRUE, 1, 0...)","Credit Card",m.sum(),"Standardised to Yes / No",df.loc[m,"Customer ID"])
flag(m,"Credit Card flag standardised"); df["Credit Card"] = c
print(df["Gender"].value_counts().to_dict(), df["Credit Card"].value_counts().to_dict())

## 3. Impossible or invalid values
Rule: set to missing, then impute with the **median of the customer's employment group** (or overall median) so the row is kept. Each is flagged.

In [ ]:
def fix(col, bad, how, step, issue, group=None):
    global df
    m = bad(df[col])
    ids = df.loc[m,"Customer ID"]
    df.loc[m,col] = np.nan
    med = df.groupby(group)[col].transform("median") if group else df[col].median()
    df[col] = df[col].fillna(med)
    record(step, issue, col, m.sum(), how, ids)
    flag(m, f"{col} invalid->imputed")

fix("Age", lambda s:(s<18)|(s>90), "Set to missing, imputed median by employment status","3a","Impossible age (5, 7, 150...)","Employment Status")
fix("Estimated Monthly Income (LKR)", lambda s:s<=0, "Set to missing, imputed median by employment status","3b","Zero / negative income","Employment Status")
fix("Last Transaction Date (Days Since)", lambda s:(s<0)|(s>365), "Set to missing, imputed overall median","3c","Negative or >365 days since last transaction")

# Negative savings balance: could be a genuine overdraft, so keep but flag
m = df["Savings Account Balance (LKR)"] < 0
df["Overdrawn_Flag"] = np.where(m,"Yes","No")
record("3d","Negative savings balance","Savings Account Balance (LKR)",m.sum(),"Kept (may be genuine overdraft); added Overdrawn_Flag",df.loc[m,"Customer ID"])
flag(m,"Overdrawn")

## 4. Missing values
Two kinds: **structural blanks** (the customer simply does not hold the product, so blank means 0 / None) and **genuine missing data**.

In [ ]:
# 4a. Missing transaction block => no recorded activity
tx_cols = [c for c in df.columns if c.startswith(("Monthly Credit","Monthly Debit","Bill Payments","ATM","POS","Total Transactions","Transaction Days","Largest Single","CC Payments - Count"))]
m = df["Total Transactions per Month"].isna()
df.loc[m,tx_cols] = 0
record("4a","Entire monthly transaction block blank","Transaction fields (12 cols)",m.sum(),"Treated as zero recorded activity (47 of 69 already flagged dormant)",df.loc[m,"Customer ID"])
flag(m,"No transaction data (set to 0)")

# 4b. Structural blanks: blank because the product is not held
def struct(flagcol, no_value, cols, fill, step):
    m = df[flagcol].eq(no_value)
    for c in cols:
        n = (m & df[c].isna()).sum()
        df.loc[m & df[c].isna(), c] = fill
        record(step,"Structural blank (product not held)",c,n,f"Filled with {fill!r}")
struct("Credit Card","No",["Credit Card Limit (LKR)","Credit Card Utilisation %"],0,"4b")
struct("Fixed Deposit","No",["FD Amount (LKR)","FD Tenure (Months)"],0,"4c")
struct("Loan Product","No",["Loan Outstanding Balance (LKR)"],0,"4d")
struct("Loan Product","No",["Loan Type"],"Not held","4e")
struct("Insurance Product","No",["Insurance Type"],"Not held","4f")
struct("Unit Trust / Investment","No",["Investment Amount (LKR)"],0,"4g")
struct("Salary Credit Flag","No",["Salary Credit Amount (LKR)"],0,"4h")

# 4i. Sparse transaction types: blank = no such transaction that month
for c in ["CC Payments - Amount (LKR)","Inward Remittances - Amount (LKR)","Interbank Transfers Out (LKR)"]:
    n = df[c].isna().sum(); df[c] = df[c].fillna(0)
    record("4i","Blank = no such transaction",c,n,"Filled with 0")

# 4j. Genuine missing: contact channel -> 'Unknown'; NPS left missing (survey non-response, not imputed)
n = df["Preferred Contact Channel"].isna().sum(); df["Preferred Contact Channel"] = df["Preferred Contact Channel"].fillna("Unknown")
record("4j","Missing contact channel","Preferred Contact Channel",n,"Labelled 'Unknown'")
record("4k","Missing NPS (survey non-response)","NPS Score (0-10)",df["NPS Score (0-10)"].isna().sum(),"Left missing; NOT imputed (would invent sentiment). Excluded from scoring")

## 5. Logical consistency checks

In [ ]:
# 5a. Credit card flag vs limit
has_limit = df["Credit Card Limit (LKR)"] > 0
m = (df["Credit Card"].eq("Yes")) & ~has_limit
record("5a","Card holder with no limit recorded","Credit Card Limit (LKR)",m.sum(),"Flagged only; limit unknown",df.loc[m,"Customer ID"]); flag(m,"CC limit missing")

# 5b. Account Age should never exceed Customer Tenure
m = df["Account Age (Months)"] > df["Customer Tenure (Months)"]
df["Tenure_Conflict"] = np.where(m,"Yes","No")
record("5b","Account Age > Customer Tenure (logically impossible)","Account Age / Tenure",m.sum(),
       "Kept both, flagged. Use Customer Tenure as the relationship-length measure; Account Age excluded from scoring",df.loc[m,"Customer ID"])

# 5c. Total Products Held should equal savings (1) + other products
yes = lambda c: df[c].eq("Yes").astype(int)
calc = 1 + yes("Credit Card")+yes("Fixed Deposit")+yes("Loan Product")+yes("Insurance Product")+yes("Unit Trust / Investment")
m = calc != df["Total Products Held"]
record("5c","Total Products Held inconsistent with product flags","Total Products Held",m.sum(),"Recomputed from flags" if m.sum() else "None found: consistent")
df["Total Products Held"] = calc

# 5d. Dormancy flag reliability (informational: we build our own active/inactive definition)
m = df["Dormancy Flag"].eq("Yes") & (df["Last Transaction Date (Days Since)"]<=90)
record("5d","Dormancy Flag contradicts recent transaction (<=90 days)","Dormancy Flag",m.sum(),"Not changed. Flag deemed unreliable; own active/inactive definition built in Step 2",df.loc[m,"Customer ID"])

## 6. Save outputs

In [ ]:
df["DQ_Flags"] = df["DQ_Flags"].str.rstrip("; ")
log = pd.DataFrame(LOG)
df.to_csv(OUT/"cleaned_customers.csv", index=False)
log.to_csv(OUT/"data_quality_log.csv", index=False)

print(f"Raw rows: {len(raw)}  ->  Clean rows: {len(df)}")
print("Remaining nulls:", df.isna().sum()[df.isna().sum()>0].to_dict())
print("Rows with at least one DQ flag:", (df['DQ_Flags']!='').sum())
log